# Goal: import annotated anndata object (from proseg segmentation)
# Then export the segmentation mask and metadata for QuPath analysis

In [1]:
# Load packages
import sys
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import scanpy as sc
import squidpy as sq
import spatialdata as sd
import matplotlib as mpl
import seaborn as sns

module_path = '/labs/delitto/james/functions/'
sys.path.append(module_path)
import jpasq

In [2]:
# Version control
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)
print("spatialdata:", sd.__version__)
print("squidpy:", sq.__version__)
print("seaborn:", sns.__version__)

sc.settings.n_jobs = -1  # Use all available cores

# Set this environment variable to tell the proj package where to look for proj.db. This gets rid of an error.
os.environ['PROJ_LIB'] = '/home/jpagolia/miniforge3/envs/jpa_squidpy/share/proj'
os.environ['PROJ_DATA'] = '/home/jpagolia/miniforge3/envs/jpa_squidpy/share/proj'

pandas: 2.3.2
numpy: 2.2.6
scanpy: 1.11.4
spatialdata: 0.7.2
squidpy: 1.8.1
seaborn: 0.13.2


In [3]:
CURRENT_DIR = Path.cwd()
PARENT_DIR = CURRENT_DIR.parent
OUTPUT_MASTER_DIR = jpasq.create_output_dir(PARENT_DIR, 'QuPath', change_dir=True)
DATA_DIR = PARENT_DIR.parent.parent / 'G4X/G4X_raw'
ANNDATA_DIR = PARENT_DIR / 'objects'

concat = sc.read_h5ad(ANNDATA_DIR / 'g4x_all_with_leiden0_8.h5ad')
concat

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath
Working directory and scanpy figure output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath


AnnData object with n_obs × n_vars = 1983386 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', 'Sample', 'ct'
    obsm: 'X_spatial'

In [4]:
palette = sns.color_palette("hls", concat.obs['ct'].cat.categories.size)
concat.uns['ct_colors'] = palette

In [5]:
print(palette)

[(0.86, 0.3712, 0.33999999999999997), (0.86, 0.49119999999999997, 0.33999999999999997), (0.86, 0.6112, 0.33999999999999997), (0.86, 0.7312000000000001, 0.33999999999999997), (0.86, 0.8512000000000001, 0.33999999999999997), (0.7487999999999997, 0.86, 0.33999999999999997), (0.6287999999999998, 0.86, 0.33999999999999997), (0.5087999999999996, 0.86, 0.33999999999999997), (0.38880000000000003, 0.86, 0.33999999999999997), (0.33999999999999997, 0.86, 0.4112), (0.33999999999999997, 0.86, 0.5312000000000001), (0.33999999999999997, 0.86, 0.6512000000000002), (0.33999999999999997, 0.86, 0.7712000000000001), (0.33999999999999997, 0.8287999999999999, 0.86), (0.33999999999999997, 0.7087999999999995, 0.86), (0.33999999999999997, 0.5887999999999997, 0.86), (0.33999999999999997, 0.4687999999999997, 0.86), (0.33999999999999997, 0.3487999999999998, 0.86), (0.4511999999999997, 0.33999999999999997, 0.86), (0.5711999999999999, 0.33999999999999997, 0.86), (0.6912000000000003, 0.33999999999999997, 0.86), (0.8

In [6]:
# Make sure cells are labeled appropriately {section}_{number} to match the segmentation mask
concat.obs_names=concat.obs['cell_name']
concat.obs

,cell,original_cell_id,centroid_x,centroid_y,centroid_z,component,volume,surface_area,scale,Section,cell_name,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,n_counts,n_genes,Patient,Sample,ct
cell_name,,,,,,,,,,,,,,,,,,,,
A02_12,12,2475,252.675415,1249.801025,0.5,7,265.0,274.0,1.0,A02,A02_12,67,4.219508,321,5.774552,321,67,3,03,0
A02_13,13,2310,262.766663,1238.970703,0.5,5,178.0,202.0,1.0,A02,A02_13,44,3.806662,284,5.652489,284,44,3,03,0
A02_14,14,3099,249.195526,1266.145142,0.5,5,181.0,220.0,1.0,A02,A02_14,46,3.850148,337,5.823046,337,46,3,03,0
A02_15,15,2926,258.255554,1259.693481,0.5,5,59.0,88.0,1.0,A02,A02_15,27,3.332205,118,4.779123,118,27,3,03,0
A02_16,16,3379,251.254486,1275.676025,0.5,7,77.0,120.0,1.0,A02,A02_16,35,3.583519,157,5.062595,157,35,3,03,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
H03_17501,17501,17039,4114.379395,4634.812988,0.5,3,207.0,264.0,1.0,H03,H03_17501,52,3.970292,389,5.966147,389,52,8,09,15
H03_17502,17502,17036,4122.844238,4624.632324,0.5,8,349.0,486.0,1.0,H03,H03_17502,29,3.401197,301,5.710427,301,29,8,09,15
H03_17503,17503,17105,4150.432617,4656.590820,0.5,4,207.0,206.0,1.0,H03,H03_17503,36,3.610918,170,5.141664,170,36,8,09,Macrophage


In [7]:
def export_qupath_from_concat(
    concat,
    base_dir=DATA_DIR,
    out_geo_dir=OUTPUT_MASTER_DIR,
    out_meta_dir=OUTPUT_MASTER_DIR,
    zarr_rel="segmentation/proseg_no_igj/proseg-output.zarr", # where shapes live
    sample_key="Section", # adata.obs label for individual sample (section)
    cell_type_key="ct", # adata.obs label for cell type
    use_layer=None, # use this layer for expression in TSV; set None to use .X
    geometry_key="cell_boundaries", # layer name in spatialdata.zarr for polygons
    id_col_candidates=("cell_name", "cell"), # how IDs are stored in shapes
    verbose=True,
):
    """
    For each sample in concat.obs[sample_key]:
      - subset concat to that sample
      - intersect with shapes from the sample's Zarr by cell IDs
      - write:
          1) <sample>_proseg_cells.geojson
          2) <sample>_lognorm_expression.tsv.gz  (meta + expression from 'use_layer' or .X)
    Assumes 'concat' already contains all preprocessing and that obs_names are the desired global cell IDs.
    """

    # --- utils ---
    def to_hex(c):
        try:
            return mpl.colors.to_hex(c)
        except Exception:
            return "#808080"
            
    def ensure_cat(s: pd.Series) -> pd.Categorical: # shows this will return a categorical
        if not isinstance(s.dtype, pd.CategoricalDtype):
            return s.astype("category")
        return s
        
    # --- set up output directories ---
    out_geo_dir  = Path(out_geo_dir);  out_geo_dir.mkdir(parents=True, exist_ok=True)
    out_meta_dir = Path(out_meta_dir); out_meta_dir.mkdir(parents=True, exist_ok=True)
    
    # --- categories & palette from concat ---
    if cell_type_key in concat.obs:
        concat.obs[cell_type_key] = ensure_cat(concat.obs[cell_type_key])
        cats = list(concat.obs[cell_type_key].cat.categories)
    else:
        cats = []
    palette = concat.uns.get(f"{cell_type_key}_colors", None)
    if palette is not None:
        # ensure it's a Python list
        if not isinstance(palette, list):
            palette = list(palette)
    if cats:
        if palette is None or len(palette) != len(cats):
            raise ValueError(
                f"Palette length mismatch for '{cell_type_key}': "
                f"{len(palette) if palette is not None else 0} vs {len(cats)} categories."
            )
        cat2color = {cat: col for cat, col in zip(cats, palette)}
    else:
        cat2color = {}
        
    # --- sample list (stable order via categorical when present) ---
    if sample_key not in concat.obs:
        raise KeyError(f"'{sample_key}' not found in concat.obs")
    if not isinstance(concat.obs[sample_key].dtype, pd.CategoricalDtype):
        samples = list(pd.Series(concat.obs[sample_key]).astype("category").cat.categories)
    else:
        samples = list(concat.obs[sample_key].cat.categories)
        
    # --- iterate samples ---
    written = []
    for sample in samples:
        sub = concat[concat.obs[sample_key] == sample].copy()
        if sub.n_obs == 0:
            if verbose: print(f"[SKIP] {sample}: no cells in concat")
            continue
            
        # enforce categorical alignment & colors on the slice
        if cats:
            sub.obs[cell_type_key] = sub.obs[cell_type_key].astype("category").cat.set_categories(cats)
            sub.uns[f"{cell_type_key}_colors"] = list(palette)
            
        # figure out zarr path for shapes
        pattern = str(Path(base_dir) / "g4*" / sample / zarr_rel)
        matches = glob.glob(pattern)
        if len(matches) == 0:
            if verbose: print(f"[WARN] {sample}: no zarr found for pattern {pattern} — GeoJSON will be skipped.")
            gdf = None
        else:
            if len(matches) > 1:
                matches.sort(key=lambda p: os.path.getmtime(p), reverse=True)
                if verbose: print(f"[INFO] {sample}: multiple zarr matches, using newest: {matches[0]}")
            zarr_path = matches[0]
            if verbose: print(f"[LOAD] {sample}: {zarr_path}")
            sdata = sd.read_zarr(zarr_path)
            if geometry_key not in sdata.shapes:
                raise KeyError(f"[ERR] {sample}: shapes layer '{geometry_key}' not found in {zarr_path}")
            gdf = sdata.shapes[geometry_key].copy()
            
        # IDs present in concat slice
        keep_ids = set(sub.obs_names.astype(str))
        
        # prepare expression matrix (use layer if present, else X)
        if use_layer is not None:
            if use_layer not in sub.layers:
                raise KeyError(f"[ERR] {sample}: requested layer '{use_layer}' not found in sub.layers")
            expr = sub.layers[use_layer]
        else:
            expr = sub.X
            
        # convert to dense/sparse DataFrame explicitly (zeros explicit for TSV)
        if hasattr(expr, "toarray"):
            expr_df = pd.DataFrame.sparse.from_spmatrix(expr, index=sub.obs_names, columns=sub.var_names)
        else:
            expr_df = pd.DataFrame(expr, index=sub.obs_names, columns=sub.var_names)
            
        # meta columns
        # (include component/surface_area if present; add color per cell_type if available)
        meta_cols = []
        for c in ("component", "surface_area"):
            if c in sub.obs.columns:
                meta_cols.append(c)
                
        # add cell_type and cell_color if available
        if cell_type_key in sub.obs.columns:
            sub.obs["cell_color"] = sub.obs[cell_type_key].map(lambda x: to_hex(cat2color.get(x))) if cat2color else "#808080"
            meta_cols = [cell_type_key, "cell_color"] + meta_cols
            
        # always include an explicit cell_name column (equal to obs_names)
        sub.obs["cell_name"] = sub.obs_names.astype(str)
        meta_cols = ["cell_name"] + meta_cols
        meta_df = sub.obs.loc[:, [c for c in meta_cols if c in sub.obs.columns]].copy()
        
        # write TSV (meta + expression)
        export_df = pd.concat([meta_df.reset_index(drop=True), expr_df.reset_index(drop=True)], axis=1)
        # enforce float32 on expression block
        export_df[expr_df.columns] = export_df[expr_df.columns].astype("float32")
        out_expr = out_meta_dir / f"{sample}_lognorm_expression.tsv.gz" if use_layer == "lognorm" \
                   else out_meta_dir / f"{sample}_expression.tsv.gz"
        export_df.to_csv(out_expr, sep="\t", index=False)
        if verbose: print(f"[ok] Wrote expression+meta → {out_expr}")
            
        # write GeoJSON (if shapes available)
        if gdf is not None:
            # pick the ID column present in shapes and normalize to 'cell_name'
            id_col = None
            for cand in id_col_candidates:
                if cand in gdf.columns:
                    id_col = cand
                    break
            if id_col is None:
                raise KeyError(f"[ERR] {sample}: shapes missing any of {id_col_candidates}")
            # if shapes use raw per-slide IDs (e.g., 'cell'), prefix with sample_ to match concat
            if id_col == "cell" and "cell_name" not in gdf.columns:
                gdf["cell_name"] = gdf["cell"].astype(str).radd(f"{sample}_")
            elif id_col == "cell_name":
                # ensure string
                gdf["cell_name"] = gdf["cell_name"].astype(str)
            else:
                # ensure we end up with cell_name column
                gdf["cell_name"] = gdf[id_col].astype(str)
            gdf = gdf[gdf["cell_name"].isin(keep_ids)].copy()
            # attach attributes
            merge_cols = ["cell_name"]
            if cell_type_key in sub.obs.columns:
                merge_cols += [cell_type_key, "cell_color"]
            if "surface_area" in sub.obs.columns: merge_cols.append("surface_area")
            if "component" in sub.obs.columns:   merge_cols.append("component")
            attr_df = sub.obs[merge_cols].copy().reset_index(drop=True)
            gdf = gdf[["cell_name", gdf.geometry.name]].merge(attr_df, on="cell_name", how="inner")
            geo_path = out_geo_dir / f"{sample}_proseg_cells.geojson"
            gdf.to_file(geo_path, driver="GeoJSON")
            if verbose: print(f"[ok] Wrote filtered GeoJSON → {geo_path}")
        else:
            if verbose: print(f"[WARN] {sample}: skipped GeoJSON (no shapes zarr found)")
        written.append(sample)
        
    if verbose:
        print(f"[done] Exported {len(written)}/{len(samples)} samples.")
    return written

In [8]:
written = export_qupath_from_concat(concat)

[LOAD] A02: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/A02/segmentation/proseg_no_igj/proseg-output.zarr


/local/scratch/jpagolia/slrmtmp.51389158/ipykernel_38138/3529495750.py:94: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(zarr_path)


[ok] Wrote expression+meta → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/A02_expression.tsv.gz
[ok] Wrote filtered GeoJSON → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/A02_proseg_cells.geojson
[LOAD] A03: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/A03/segmentation/proseg_no_igj/proseg-output.zarr


/local/scratch/jpagolia/slrmtmp.51389158/ipykernel_38138/3529495750.py:94: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(zarr_path)


[ok] Wrote expression+meta → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/A03_expression.tsv.gz
[ok] Wrote filtered GeoJSON → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/A03_proseg_cells.geojson
[LOAD] A04: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L004_424ec4f5bf7d29d9e/A04/segmentation/proseg_no_igj/proseg-output.zarr


...

[ok] Wrote expression+meta → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/H01_expression.tsv.gz
[ok] Wrote filtered GeoJSON → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/H01_proseg_cells.geojson
[LOAD] H02: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L002_489c8745a6d2d69ca/H02/segmentation/proseg_no_igj/proseg-output.zarr


/local/scratch/jpagolia/slrmtmp.51389158/ipykernel_38138/3529495750.py:94: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(zarr_path)


[ok] Wrote expression+meta → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/H02_expression.tsv.gz
[ok] Wrote filtered GeoJSON → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/H02_proseg_cells.geojson
[LOAD] H03: /oak/stanford/groups/longaker/ULMS/G4X/G4X_raw/g4-033-076-FC4-L003_472c4f0ef9f2af777/H03/segmentation/proseg_no_igj/proseg-output.zarr


/local/scratch/jpagolia/slrmtmp.51389158/ipykernel_38138/3529495750.py:94: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(zarr_path)


[ok] Wrote expression+meta → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/H03_expression.tsv.gz
[ok] Wrote filtered GeoJSON → /oak/stanford/groups/longaker/ULMS/revision/G4X/QuPath/H03_proseg_cells.geojson
[done] Exported 26/26 samples.


In [9]:
print(concat.obs['ct'].cat.categories)

Index(['0', '1', '10', '11', '12', '13', '14', '15', '16', '2', '3', '4', '5',
       '6', '7', '8', '9', 'Adipocyte', 'B', 'Endothelial', 'Fibroblast',
       'Macrophage', 'Mast', 'Necrosis', 'Pericyte', 'T_and_NK'],
      dtype='object')
